In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

columnas = [
    "estado_cuenta", "duracion_meses", "historial_credito", "proposito",
    "monto_credito", "ahorros", "empleo_actual", "tasa_pago_pct_ingreso",
    "estado_civil_sexo", "deudores_garantes", "residencia_actual_anos",
    "propiedad", "edad", "otros_planes_pago", "vivienda",
    "creditos_existentes", "trabajo", "personas_a_cargo",
    "telefono", "trabajador_extranjero", "target"
]

df = pd.read_csv("../data/raw/german.data", sep=" ", header=None, names=columnas)
df["target"] = (df["target"] == 2).astype(int)
print(df.shape)

(1000, 21)


In [2]:
# Mapeo según german.doc
mapeo = {
    "estado_cuenta": {
        "A11": "saldo_negativo",
        "A12": "saldo_0_a_200",
        "A13": "saldo_mayor_200",
        "A14": "sin_cuenta_corriente",
    },
    "historial_credito": {
        "A30": "sin_creditos_o_pagados",
        "A31": "todos_pagados_aqui",
        "A32": "creditos_al_dia",
        "A33": "retraso_en_pasado",
        "A34": "cuenta_critica_otros_bancos",
    },
    "proposito": {
        "A40": "carro_nuevo",
        "A41": "carro_usado",
        "A42": "muebles_equipos",
        "A43": "radio_television",
        "A44": "electrodomesticos",
        "A45": "reparaciones",
        "A46": "educacion",
        "A47": "vacaciones",
        "A48": "reentrenamiento",
        "A49": "negocio",
        "A410": "otros",
    },
    "ahorros": {
        "A61": "menos_100",
        "A62": "100_a_500",
        "A63": "500_a_1000",
        "A64": "mayor_1000",
        "A65": "desconocido_sin_ahorros",
    },
    "empleo_actual": {
        "A71": "desempleado",
        "A72": "menos_1_ano",
        "A73": "1_a_4_anos",
        "A74": "4_a_7_anos",
        "A75": "mayor_7_anos",
    },
    "estado_civil_sexo": {
        "A91": "hombre_divorciado",
        "A92": "mujer_divorciada_casada",
        "A93": "hombre_soltero",
        "A94": "hombre_casado_viudo",
        "A95": "mujer_soltera",
    },
    "deudores_garantes": {
        "A101": "ninguno",
        "A102": "codeudor",
        "A103": "garante",
    },
    "propiedad": {
        "A121": "bienes_raices",
        "A122": "seguro_o_ahorro_construccion",
        "A123": "carro_u_otro",
        "A124": "sin_propiedad",
    },
    "otros_planes_pago": {
        "A141": "banco",
        "A142": "tiendas",
        "A143": "ninguno",
    },
    "vivienda": {
        "A151": "alquilada",
        "A152": "propia",
        "A153": "gratis",
    },
    "trabajo": {
        "A171": "no_calificado_no_residente",
        "A172": "no_calificado_residente",
        "A173": "calificado",
        "A174": "altamente_calificado",
    },
    "telefono": {
        "A191": "sin_telefono",
        "A192": "con_telefono",
    },
    "trabajador_extranjero": {
        "A201": "si",
        "A202": "no",
    },
}

for col, dic in mapeo.items():
    df[col] = df[col].map(dic)

df.head()

,estado_cuenta,duracion_meses,historial_credito,proposito,monto_credito,ahorros,empleo_actual,tasa_pago_pct_ingreso,estado_civil_sexo,deudores_garantes,...,propiedad,edad,otros_planes_pago,vivienda,creditos_existentes,trabajo,personas_a_cargo,telefono,trabajador_extranjero,target
0,saldo_negativo,6,cuenta_critica_otros_bancos,radio_television,1169,desconocido_sin_ahorros,mayor_7_anos,4,hombre_soltero,ninguno,...,bienes_raices,67,ninguno,propia,2,calificado,1,con_telefono,si,0
1,saldo_0_a_200,48,creditos_al_dia,radio_television,5951,menos_100,1_a_4_anos,2,mujer_divorciada_casada,ninguno,...,bienes_raices,22,ninguno,propia,1,calificado,1,sin_telefono,si,1
2,sin_cuenta_corriente,12,cuenta_critica_otros_bancos,educacion,2096,menos_100,4_a_7_anos,2,hombre_soltero,ninguno,...,bienes_raices,49,ninguno,propia,1,no_calificado_residente,2,sin_telefono,si,0
3,saldo_negativo,42,creditos_al_dia,muebles_equipos,7882,menos_100,4_a_7_anos,2,hombre_soltero,garante,...,seguro_o_ahorro_construccion,45,ninguno,gratis,1,calificado,2,sin_telefono,si,0
4,saldo_negativo,24,retraso_en_pasado,carro_nuevo,4870,menos_100,1_a_4_anos,3,hombre_soltero,ninguno,...,sin_propiedad,53,ninguno,gratis,2,calificado,2,sin_telefono,si,1


In [6]:
# Variables con orden natural claro (ordinales)
ordinales = {
    "estado_cuenta": [
        "saldo_negativo",        # peor
        "saldo_0_a_200",
        "saldo_mayor_200",
        "sin_cuenta_corriente",  # mejor (no usa cuenta corriente = sin deudas asociadas)
    ],
    "ahorros": [
        "menos_100",
        "desconocido_sin_ahorros", 
        "100_a_500",
        "500_a_1000",
        "mayor_1000",
    ],
    "empleo_actual": [
        "desempleado",
        "menos_1_ano",
        "1_a_4_anos",
        "4_a_7_anos",
        "mayor_7_anos",
    ],
    "trabajo": [
        "no_calificado_no_residente",
        "no_calificado_residente",
        "calificado",
        "altamente_calificado",
    ],
}

# Variables sin orden (nominales) → one-hot
nominales = [
    "historial_credito",      
    "proposito",
    "estado_civil_sexo",
    "deudores_garantes",
    "propiedad",
    "otros_planes_pago",
    "vivienda",
    "telefono",
    "trabajador_extranjero",
]

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,         
    random_state=42
)

print(f"Train: {X_train.shape}, distribución target: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Test:  {X_test.shape}, distribución target: {y_test.value_counts(normalize=True).round(3).to_dict()}")

Train: (800, 20), distribución target: {0: 0.7, 1: 0.3}
Test:  (200, 20), distribución target: {0: 0.7, 1: 0.3}


In [ ]:
# Listas de columnas por tipo
cols_numericas = [
    "duracion_meses", "monto_credito", "tasa_pago_pct_ingreso",
    "residencia_actual_anos", "edad", "creditos_existentes", "personas_a_cargo"
]
cols_ordinales = list(ordinales.keys())
cols_nominales = nominales

# Categorías ordenadas (en el mismo orden que las cols_ordinales)
categorias_ordinales = [ordinales[col] for col in cols_ordinales]

preprocesador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), cols_numericas),
        ("ord", OrdinalEncoder(categories=categorias_ordinales), cols_ordinales),
        ("nom", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cols_nominales),
    ],
    remainder="drop"
)

# Fit solo con train
preprocesador.fit(X_train)

X_train_proc = preprocesador.transform(X_train)
X_test_proc = preprocesador.transform(X_test)

nombres_num = cols_numericas
nombres_ord = cols_ordinales
nombres_nom = preprocesador.named_transformers_["nom"].get_feature_names_out(cols_nominales).tolist()
nombres_finales = nombres_num + nombres_ord + nombres_nom

print(f"Shape X_train procesado: {X_train_proc.shape}")
print(f"Shape X_test procesado:  {X_test_proc.shape}")
print(f"Total de features tras encoding: {len(nombres_finales)}")
print(f"\nPrimeras 10 columnas resultantes:")
for n in nombres_finales[:10]:
    print(f"  - {n}")

Shape X_train procesado: (800, 38)
Shape X_test procesado:  (200, 38)
Total de features tras encoding: 38

Primeras 10 columnas resultantes:
  - duracion_meses
  - monto_credito
  - tasa_pago_pct_ingreso
  - residencia_actual_anos
  - edad
  - creditos_existentes
  - personas_a_cargo
  - estado_cuenta
  - ahorros
  - empleo_actual


In [9]:
# Convertir a DataFrame para ver mejor
X_train_proc_df = pd.DataFrame(X_train_proc, columns=nombres_finales)
X_train_proc_df.head()

,duracion_meses,monto_credito,tasa_pago_pct_ingreso,residencia_actual_anos,edad,creditos_existentes,personas_a_cargo,estado_cuenta,ahorros,empleo_actual,...,deudores_garantes_ninguno,propiedad_carro_u_otro,propiedad_seguro_o_ahorro_construccion,propiedad_sin_propiedad,otros_planes_pago_ninguno,otros_planes_pago_tiendas,vivienda_gratis,vivienda_propia,telefono_sin_telefono,trabajador_extranjero_si
0,1.289592,1.925766,0.052281,1.053413,1.057784,-0.718745,-0.436436,0.0,1.0,4.0,...,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0
1,-0.742595,-0.892853,0.942164,1.053413,0.242449,-0.718745,-0.436436,3.0,0.0,4.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
2,1.289592,0.816433,0.052281,-0.747293,-0.663479,1.045047,-0.436436,0.0,0.0,2.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0
3,1.289592,0.299568,0.052281,-0.747293,-0.572886,-0.718745,-0.436436,1.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
4,2.305686,1.982655,-1.727486,-0.747293,-1.025850,-0.718745,-0.436436,1.0,1.0,3.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0


In [10]:
import joblib

# Guardamos el preprocesador entrenado para reutilizar
joblib.dump(preprocesador, "../data/processed/preprocesador.joblib")

# Guardamos también los datos procesados como CSV (útil para inspección)
pd.DataFrame(X_train_proc, columns=nombres_finales).to_csv(
    "../data/processed/X_train.csv", index=False
)
pd.DataFrame(X_test_proc, columns=nombres_finales).to_csv(
    "../data/processed/X_test.csv", index=False
)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Preprocesador y datos guardados en data/processed/")

Preprocesador y datos guardados en data/processed/
